In [1]:
# Stream India Data Generation Script 
# Project : Subcsriber Churn Intelligence System 
# Notebook 1 : StreamIndia_Data_Generation.ipynb

In [3]:
# importing necessary libraries for data generation 
import pandas as pd 
import numpy as np 
from faker import Faker 
import random 
from datetime import datetime , timedelta 
import os 

In [5]:
!pip install faker 


[notice] A new release of pip is available: 23.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# SUBSCIBERS DATABASE CREATION TABLE 1 : SUBSCRIBERS.CSV

In [7]:
fake = Faker ('en_IN') #Indian loace 
np.random.seed(42)
random.seed(42)

In [8]:
# TABLE 1 : SUBSCRIBERS ( 15K ROWS )

In [9]:
NUM_USERS = 15000

def generate_subscribers(n):
    user_ids = [f"USR{str(i).zfill(6)}" for i in range (1,n+1)]
    states = ['Maharashtra','Karnataka','Tamil Nadu','Delhi','Uttar Pradesh','Gujarat','West Bengal','Telangana','Rajashthan','Kerala','Madhya Pradesh','Punjab','Bihar','Haryana','Odhisha']
    state_weights = [0.14,0.10,0.09,0.10,0.08,0.07,0.07,0.07,0.05,0.05,0.04,0.04,0.04,0.03,0.03]
    subscription_plans = ['Mobile','Basic','Standard','Premium']
    plan_price_map = {'Mobile': 149, 'Basic': 299, 'Standard': 499, 'Premium':799}
    plan_weights = [0.35,0.25,0.25,0.15]
    devices = ['Mobile','Smart TV','Laptop','Tablet','Fire Stick']
    device_weights = [0.50,0.20,0.18,0.07,0.05]
    acquisition_channels = ['Organic Search','Social Media','Referral','Paid Ad','Telecom Bundle','App Store']
    channel_weights = [0.10,0.25,0.15,0.20,0.15,0.05]

    #Registeration dates spread over 3 years 
    start_date = datetime(2022,1,1)
    end_date = datetime(2024,12,31)
    date_range = (end_date - start_date).days 
    records = []
    for uid in user_ids:
        plan = random.choices(subscription_plans, plan_weights)[0]
        primary_device = random.choices(devices,device_weights)[0]
        channel = random.choices(acquisition_channels,channel_weights)[0]
        state = random.choices(states, state_weights)[0]
        age = random.randint(18,55)
        reg_date = start_date + timedelta(days=random.randint(0,date_range))

        #Churn Logic - reailistic prbabilities based on behavior signals 
        churn_prob = 0.25 # base churn rate ~ 25% 

        if plan == 'Mobile': churn_prob += 0.10
        if plan == 'Premium' : churn_prob -= 0.10
        if primary_device == 'Mobile' : churn_prob += 0.05
        if primary_device == 'Smart TV' : churn_prob -= 0.08 
        if channel == 'Telecom Bundle' : churn_prob -= 0.07 
        if channel == 'Paid Ad' : churn_prob += 0.08 
        if age < 25: churn_prob += 0.05
        if age > 40: churn_prob -= 0.05

        churn_prob = max(0.05,min(0.85, churn_prob))
        churned = 1 if random.random() < churn_prob else 0

        # Churned date - only if churned 
        if churned:
            days_active = random.randint(30,540)
            churn_date = reg_date + timedelta(days=days_active)
            churn_date = min(churn_date,end_date)
            churn_date_str = churn_date.strftime('%Y-%m-%d')
        else:
             churn_date_str = None 
        records.append({
            'user_id' : uid,
            'age': age,
            'gender' : random.choices(['Male','Female','Other'], [0.52,0.45,0.03])[0],
            'state' : state,
            'city_tier' : random.choices(['Tier 1','Tier 2','Tier 3'] , [0.35,0.40,0.25])[0],
            'registration_date' : reg_date.strftime('%Y-%m-%d'),
            'subscription_plan' : plan,
            'monthly_price_inr' : plan_price_map[plan],
            'primary_device' : primary_device,
            'acquisition_channel' : channel,
            'num_profiles' : random.randint(1,5),
            'language_preference' : random.choices(
                                                   ['Hindi','English','Marathi','Tamil','Telugu','Bengali','Kannada'],
                                                      [0.30, 0.20, 0.10, 0.12, 0.10, 0.10, 0.08])[0],
            'auto_renewal': random.choices([1, 0], weights=[0.70, 0.30])[0],
            'churned' : churned,
            'churn_date' : churn_date_str
            
        })
    return pd.DataFrame(records)

subscribers_df = generate_subscribers(NUM_USERS)
print(f"Subscribers table : {subscribers_df.shape}")
print(f"Churn Rate : {subscribers_df['churned'].mean():.2%}")
subscribers_df.head(3)

Subscribers table : (15000, 15)
Churn Rate : 27.17%


,user_id,age,gender,state,city_tier,registration_date,subscription_plan,monthly_price_inr,primary_device,acquisition_channel,num_profiles,language_preference,auto_renewal,churned,churn_date
0,USR000001,24,Male,Karnataka,Tier 1,2022-06-28,Standard,499,Mobile,Social Media,2,Marathi,1,0,None
1,USR000002,55,Male,Karnataka,Tier 3,2023-07-24,Mobile,149,Smart TV,Referral,4,English,1,0,None
2,USR000003,40,Female,Maharashtra,Tier 2,2023-06-26,Premium,799,Mobile,Organic Search,4,Hindi,1,0,None


In [11]:
subscribers_df.tail()

,user_id,age,gender,state,city_tier,registration_date,subscription_plan,monthly_price_inr,primary_device,acquisition_channel,num_profiles,language_preference,auto_renewal,churned,churn_date
14995,USR014996,31,Female,Madhya Pradesh,Tier 3,2024-07-29,Mobile,149,Laptop,Social Media,3,Tamil,1,1,2024-12-31
14996,USR014997,41,Male,Maharashtra,Tier 2,2023-07-12,Standard,499,Laptop,Social Media,4,English,0,0,None
14997,USR014998,48,Female,Telangana,Tier 3,2023-03-24,Mobile,149,Laptop,Paid Ad,4,English,1,0,None
14998,USR014999,53,Male,Karnataka,Tier 3,2024-11-23,Mobile,149,Tablet,Referral,2,Telugu,1,0,None
14999,USR015000,42,Male,Telangana,Tier 3,2023-07-23,Premium,799,Mobile,Telecom Bundle,4,Telugu,1,0,None


In [16]:
subscribers_df.to_csv('subscribers.csv', index=False)

# CREATION OF TABLE 2 : WATCH_HISTORY.CSV

In [19]:
# TABLE 2 : WATCH HISTORY (1,50,000 ROWS)

In [26]:
def generate_watch_history(subscribers_df):
    genres = ['Drama','Comedy','Action','Thriller','Romance','Horror','Documentary','Sports','Kids','Reality TV']
    languages = ['Hindi','English','Marathi','Tamil','Telugu','Bengali','Kannada']
    content_types = ['Movie','Web Series','Short Film','Live Sports','Documentary']
    content_weights = [0.35,0.40,0.05,0.15,0.05]
    genre_weights = [0.18, 0.15, 0.13, 0.12, 0.10, 0.08, 0.07, 0.07, 0.05, 0.05]
    devices = ['Mobile','Smart TV','Laptop','Tablet','Fire Stick']
    records = []
    for _, user in subscribers_df.iterrows():
        if user['churned'] == 1:
            num_sessions = random.randint(5, 40)
        else:
            num_sessions = random.randint(20, 120)
        reg_date = datetime.strptime(user['registration_date'], '%Y-%m-%d')
        if user['churn_date']:
            end_date = datetime.strptime(user['churn_date'], '%Y-%m-%d')
        else:
            end_date = datetime(2024, 12, 31)
        active_days = (end_date - reg_date).days
        if active_days < 1:
            active_days = 1
        for _ in range(num_sessions):
            watch_date = reg_date + timedelta(days=random.randint(0, active_days))
            if user['churned'] == 1:
                duration = random.randint(5, 45)
            else:
                duration = random.randint(20, 180)
            records.append({
                'session_id'          : f"SES{random.randint(100000, 999999)}",
                'user_id'             : user['user_id'],
                'watch_date'          : watch_date.strftime('%Y-%m-%d'),
                'genre'               : random.choices(genres, genre_weights)[0],
                'content_type'        : random.choices(content_types, content_weights)[0],
                'language_watched'    : random.choices(languages)[0],
                'watch_duration_mins' : duration,
                'device_used'         : random.choices(devices)[0],
                'completed'           : 1 if duration > 60 else 0
            })
    return pd.DataFrame(records)

watch_history_df = generate_watch_history(subscribers_df)
print(f"Watch History table: {watch_history_df.shape}")
print(f"Avg session duration: {watch_history_df['watch_duration_mins'].mean():.1f} mins")
print(f"Completion rate: {watch_history_df['completed'].mean():.2%}")
watch_history_df.to_csv('watch_history.csv', index=False, encoding='utf-8-sig')
print("Saved!")
watch_history_df.head(3)

Watch History table: (862217, 9)
Avg session duration: 92.0 mins
Completion rate: 66.64%
Saved!


,session_id,user_id,watch_date,genre,content_type,language_watched,watch_duration_mins,device_used,completed
0,SES459983,USR000001,2023-11-19,Romance,Web Series,Hindi,111,Fire Stick,1
1,SES746094,USR000001,2022-08-06,Action,Live Sports,Marathi,100,Laptop,1
2,SES378067,USR000001,2024-10-01,Drama,Movie,Bengali,78,Laptop,1


In [23]:
watch_history_df.tail()

,session_id,user_id,watch_date,genre,content_type,language_watched,watch_duration_mins,device_used,completed
855503,SES388129,USR015000,2024-06-22,Sports,Movie,Telugu,161,Mobile,1
855504,SES608642,USR015000,2023-09-08,Horror,Live Sports,English,120,Smart TV,1
855505,SES537722,USR015000,2024-06-14,Action,Web Series,Tamil,109,Mobile,1
855506,SES988445,USR015000,2023-10-10,Action,Live Sports,Marathi,27,Tablet,0
855507,SES821741,USR015000,2023-09-22,Action,Short Film,Marathi,172,Tablet,1


# CREATION OF TABLE 3 : SUBSCRIPTION PAYMENTS.CSV

In [121]:
#TABLE 3 : SUBSCRIPTION PAYMENTS (45,000 ROWS)

In [147]:
def generate_payments(subscribers_df):
    payment_methods = ['UPI','Credit Card','Debit Card','Net Banking','Wallet']
    payment_weights = [0.45, 0.20, 0.20, 0.10, 0.05]
    records = []
    for _, user in subscribers_df.iterrows():
        reg_date = datetime.strptime(user['registration_date'], '%Y-%m-%d')
        if user['churn_date']:
            end_date = datetime.strptime(user['churn_date'], '%Y-%m-%d')
        else:
            end_date = datetime(2024, 12, 31)
        active_months = max(1, (end_date - reg_date).days // 30)
        for month in range(active_months):
            payment_date = reg_date + timedelta(days=30 * month)
            if user['churned'] == 1:
                failed = 1 if random.random() < 0.25 else 0
            else:
                failed = 1 if random.random() < 0.05 else 0
            records.append({
                'payment_id'      : f"PAY{random.randint(1000000, 9999999)}",
                'user_id'         : user['user_id'],
                'payment_date'    : payment_date.strftime('%Y-%m-%d'),
                'amount_inr'      : user['monthly_price_inr'],
                'payment_method'  : random.choices(payment_methods, payment_weights)[0],
                'payment_status'  : 'Failed' if failed else 'Success',
                'plan_at_payment' : user['subscription_plan']
            })
    return pd.DataFrame(records)

payments_df = generate_payments(subscribers_df)
print(f"Payments table: {payments_df.shape}")
print(f"Payment failure rate: {(payments_df['payment_status']=='Failed').mean():.2%}")
payments_df.to_csv('payments.csv', index=False, encoding='utf-8-sig')
print("Saved!")
payments_df.head(3)

Payments table: (223480, 7)
Payment failure rate: 7.69%
Saved!


,payment_id,user_id,payment_date,amount_inr,payment_method,payment_status,plan_at_payment
0,PAY5170401,USR000001,2022-06-28,499,UPI,Success,Standard
1,PAY4682411,USR000001,2022-07-28,499,Credit Card,Success,Standard
2,PAY1540771,USR000001,2022-08-27,499,Net Banking,Success,Standard


In [149]:
payments_df.tail()

,payment_id,user_id,payment_date,amount_inr,payment_method,payment_status,plan_at_payment
223475,PAY6306228,USR015000,2024-07-17,799,UPI,Success,Premium
223476,PAY8756045,USR015000,2024-08-16,799,Credit Card,Success,Premium
223477,PAY7673449,USR015000,2024-09-15,799,Net Banking,Success,Premium
223478,PAY8349852,USR015000,2024-10-15,799,UPI,Success,Premium
223479,PAY4740276,USR015000,2024-11-14,799,Credit Card,Success,Premium


# CREATION OF TABLE 4 : SUPPORT TICKETS .CSV 

In [152]:
# ============================================================
# TABLE 4: SUPPORT TICKETS (~18,000 rows)
# ============================================================

def generate_support_tickets(subscribers_df):
    
    issue_types = ['Buffering Issue', 'Payment Failed', 'Login Problem',
                   'Content Not Available', 'App Crash', 'Subscription Query',
                   'Audio/Video Quality', 'Cancellation Request']
    
    # Churned users more likely to raise cancellation/payment issues
    issue_weights_churned  = [0.20, 0.20, 0.10, 0.15, 0.10, 0.05, 0.05, 0.15]
    issue_weights_active   = [0.25, 0.10, 0.15, 0.20, 0.10, 0.10, 0.10, 0.00]
    
    resolutions = ['Resolved', 'Unresolved', 'Escalated']
    
    records = []
    
    for _, user in subscribers_df.iterrows():
        
        # Only 60% of churned and 20% of active users raise tickets
        if user['churned'] == 1:
            if random.random() > 0.60:
                continue
            num_tickets = random.randint(1, 4)
        else:
            if random.random() > 0.20:
                continue
            num_tickets = random.randint(1, 2)
        
        reg_date = datetime.strptime(user['registration_date'], '%Y-%m-%d')
        
        if user['churn_date']:
            end_date = datetime.strptime(user['churn_date'], '%Y-%m-%d')
        else:
            end_date = datetime(2024, 12, 31)
        
        active_days = max(1, (end_date - reg_date).days)
        
        for _ in range(num_tickets):
            ticket_date = reg_date + timedelta(days=random.randint(0, active_days))
            
            if user['churned'] == 1:
                issue = random.choices(issue_types, issue_weights_churned)[0]
                resolution = random.choices(resolutions, [0.40, 0.35, 0.25])[0]
            else:
                issue = random.choices(issue_types, issue_weights_active)[0]
                resolution = random.choices(resolutions, [0.70, 0.20, 0.10])[0]
            
            records.append({
                'ticket_id'        : f"TKT{random.randint(100000, 999999)}",
                'user_id'          : user['user_id'],
                'ticket_date'      : ticket_date.strftime('%Y-%m-%d'),
                'issue_type'       : issue,
                'resolution_status': resolution,
                'response_time_hrs': random.randint(1, 72)
            })
    
    return pd.DataFrame(records)

support_tickets_df = generate_support_tickets(subscribers_df)
print(f"Support Tickets table: {support_tickets_df.shape}")
print(f"Resolution rate: {(support_tickets_df['resolution_status']=='Resolved').mean():.2%}")
support_tickets_df.to_csv('support_tickets.csv', index=False, encoding='utf-8-sig')
print("Saved!")
support_tickets_df.head(3)

Support Tickets table: (9529, 6)
Resolution rate: 51.08%
Saved!


,ticket_id,user_id,ticket_date,issue_type,resolution_status,response_time_hrs
0,TKT642112,USR000013,2023-02-22,Audio/Video Quality,Resolved,68
1,TKT632420,USR000013,2023-06-30,Buffering Issue,Resolved,55
2,TKT313892,USR000014,2023-11-14,Cancellation Request,Resolved,59


# DATASET VALIDATION 

In [155]:
# ============================================================
# DATASET VALIDATION — Run this last
# ============================================================

print("=" * 50)
print("STREAMINDIA DATASET SUMMARY")
print("=" * 50)

print(f"\n📋 Subscribers      : {len(subscribers_df):,} rows")
print(f"🎬 Watch History    : {len(watch_history_df):,} rows")
print(f"💳 Payments         : {len(payments_df):,} rows")
print(f"🎫 Support Tickets  : {len(support_tickets_df):,} rows")

print(f"\n📊 Churn Rate       : {subscribers_df['churned'].mean():.2%}")
print(f"💰 Avg Monthly Rev  : ₹{subscribers_df['monthly_price_inr'].mean():,.0f}")
print(f"📺 Avg Watch Time   : {watch_history_df['watch_duration_mins'].mean():.1f} mins")
print(f"❌ Payment Failures : {(payments_df['payment_status']=='Failed').mean():.2%}")
print(f"🎫 Ticket Res. Rate : {(support_tickets_df['resolution_status']=='Resolved').mean():.2%}")

print(f"\n✅ All 4 tables saved as CSV")
print(f"✅ Phase 1 COMPLETE")
print("=" * 50)

STREAMINDIA DATASET SUMMARY

📋 Subscribers      : 15,000 rows
🎬 Watch History    : 854,184 rows
💳 Payments         : 223,480 rows
🎫 Support Tickets  : 9,529 rows

📊 Churn Rate       : 27.17%
💰 Avg Monthly Rev  : ₹372
📺 Avg Watch Time   : 91.9 mins
❌ Payment Failures : 7.69%
🎫 Ticket Res. Rate : 51.08%

✅ All 4 tables saved as CSV
✅ Phase 1 COMPLETE


# END OF TABLE CREATION

### English explaination of table creation for process flow 

#### TABLE 1 : SUBSCRIBERS_DF

1. We are creating a fake but realistic database of 15,000 OTT users. Each user has a profile — who they are, what plan they bought, which device they use, and most importantly — did they churn or not.

2. We don't want equal users from every state. In reality, Maharashtra and Delhi have more internet users than Bihar or Odisha. So we give higher probability to bigger states. This makes our data feel real.

3. Most Indian OTT users buy the cheapest Mobile plan. Very few buy Premium. So we give Mobile the highest weight. This mirrors real OTT subscription distribution in India.

4. We want users to have joined at different times between 2022 and 2024 — not all on the same day. So we pick a random number of days and add it to the start date. This spreads registrations realistically over 3 years.

5. We start every user with a 25% base chance of churning. Then we adjust it based on what we know about OTT user behaviour:

~ Mobile plan users churn more — they're price sensitive and often subscribed just for one show
~ Premium users churn less — they're invested and use more features
~ Mobile device users churn more — they're casual viewers
~ Smart TV users churn less — they watch as a family, more habitual
~ Telecom bundle users churn less — their subscription is tied to their phone plan
~ Paid Ad users churn more — they came because of a discount, not loyalty
~ Under 25 churn more — young users are fickle and explore multiple platforms
~ Over 40 churn less — older users are more habitual and brand loyal

This logic is based on real OTT industry patterns. This is what makes your dataset intelligent, not random.

6. After all the adjustments, we make sure churn probability never goes below 5% or above 85%. Even the most loyal user has some chance of leaving. Even the most at-risk user might stay. This keeps our data realistic.

7. This is the final churn decision. random.random() generates a number between 0 and 1. If that number is less than the user's churn probability, they churned. Otherwise they stayed. Think of it like rolling a dice weighted by everything we know about that user.

8. If a user churned, we calculate when they left. They stayed somewhere between 30 days and 540 days (1.5 years) before leaving. We also make sure the churn date never goes beyond December 2024 — our dataset's end date.

END NOTE : 
We didn't just create random users. We created users whose behaviour, plan, device, age and location all influence whether they churn — exactly like a real OTT platform's data would look.

#### TABLE 2 : WATCH HISTORY 

The Big Picture
Table 1 told us who the users are. Table 2 tells us what they actually did on the platform. Every row in this table is one viewing session — one time a user sat down and watched something on StreamIndia.
This table is the heartbeat of the project. Every insight about engagement, retention and churn behaviour will come from here.

1. We loop through every single user from Table 1 one by one. For each user we will generate their entire watch history. The _ means we don't care about the row number — we only care about the user's data.

2. This is the first piece of business logic in this table. A user who eventually churned was never deeply engaged with the platform — so we give them fewer viewing sessions, between 5 and 40. An active loyal user watches much more regularly — so they get between 20 and 120 sessions. This creates a natural pattern that our ML model will later detect — low watch frequency is a churn signal.

3. Every user's watch history should only exist within their active period. A user who joined in March 2022 and churned in August 2022 can only have watch sessions between those two dates — not before, not after. So we define their personal start and end window. If they never churned, their window goes all the way to December 2024.

4. We calculate how many days this user was active on the platform. We set a minimum of 1 day because some users might have registered and churned on the same day — and we can't generate a watch session in zero days. This is called a safety check — it prevents our code from crashing on edge cases.

5. Now for each session we pick a random date within the user's active window. If a user was active for 200 days, each session gets assigned a random day somewhere in those 200 days. This spreads their watch history naturally over time — exactly like real viewing behaviour.

6. This is the second piece of business logic. Churned users didn't just watch less frequently — they also watched for shorter durations when they did watch. Maybe they opened the app, found nothing interesting, and closed it after 10 minutes. Loyal users binge for hours. So churned users get 5 to 45 minute sessions, active users get 20 to 180 minutes. Short session duration is another churn signal.

7. This marks whether a user finished the content they started. We assume anything over 60 minutes means they completed the movie or episode. Anything under 60 minutes means they dropped off midway. Low completion rate is another churn signal — users who keep abandoning content eventually leave the platform.

8. Every session needs a unique ID just like a real database. We generate a random 6 digit number prefixed with SES. This is how real OTT platforms track individual viewing events in their logs.

9. Why This Table Has 854,184 Rows
Because we have 15,000 users and each user gets anywhere from 5 to 120 sessions. The average works out to roughly 57 sessions per user across their lifetime. 15,000 × 57 = ~855,000 rows. This is intentional — real OTT databases have millions of watch events.


The Three Churn Signals We Baked Into This Table:
SignalChurned UsersActive UsersNumber of sessions5 to 4020 to 120Session duration5 to 45 mins20 to 180 minsCompletion rateLowHigh
These patterns are what our EDA will discover and our ML model will learn from. We didn't make the data random — we made it tell a story.

The One Line Summary of Table 2:
We created a realistic viewing history where churned users naturally watch less, watch shorter, and complete less — so that when we analyse this data later, the patterns reveal themselves like they would in a real OTT platform.

#### TABLE 3 : SUBSCRIPTION PAYMENTS 

The Big Picture
Table 1 told us who the users are. Table 2 told us what they watched. Table 3 tells us how and whether they paid.This table is the money table. Every row is one monthly payment attempt by a user. This is where we capture one of the strongest churn signals in any subscription business — payment failures.In the real world, payment failure is often the last thing that happens before a user churns. Either they deliberately let their card fail because they want to cancel, or a failed payment interrupts their experience and they never come back. Either way — payment failure predicts churn.

1. We calculate how many months this user was active. We divide their active days by 30 to get months. The // means integer division — so 95 days becomes 3 months, not 3.16. We set a minimum of 1 month because every user made at least one payment when they subscribed. This is another safety check just like in Table 2.

2. Every month we generate one payment attempt for that user. If a user was active for 8 months, they get 8 payment rows — one every 30 days from their registration date. This mimics how real subscription billing works — your card gets charged every 30 days automatically.

3. This is the core business logic of this table. Churned users have a 25% chance of any given payment failing. Active loyal users only have a 5% chance of payment failure. This creates a very realistic pattern — churned users have a history of payment problems before they finally leave. Active users almost always pay successfully.
Think about it from real life — when you want to cancel Netflix, you might stop the auto-renewal, or your card mysteriously fails the month before you cancel. This behaviour is captured here.

4. Simple — if the payment failed variable is 1, we label it Failed. Otherwise Success. This becomes our key column for payment failure analysis later in SQL and EDA.

5. We record which plan the user was on at the time of payment. This is important because in real OTT platforms users downgrade their plan before churning — they go from Premium to Mobile and then cancel. Recording the plan at each payment lets us detect plan downgrade patterns which is another churn signal.

6. We assign a payment method to each transaction. UPI has the highest weight at 45% because in India UPI is the dominant payment method. Credit card and debit card share 20% each. This reflects real Indian digital payment behaviour in 2022 to 2024.

7. Why This Table Has 223,480 Rows
Each user gets one payment row per active month. Users were active for an average of roughly 15 months across our 3 year window. 15,000 users × 15 months = ~225,000 rows. This matches our output perfectly.

The Two Churn Signals We Baked Into This Table:
SignalChurned UsersActive UsersPayment failure rate25% per month5% per monthTotal payments madeFewer months activeMore months active

The One Line Summary of Table 3:
We created a payment history where churned users have significantly more payment failures than active users — because in real subscription businesses, payment failure is one of the strongest and most reliable predictors of churn.


#### TABLE 4 : SUPPORT TICKETS 

The Big Picture
Table 1 — who they are.
Table 2 — what they watched.
Table 3 — how they paid.
Table 4 — when they complained.This table captures every time a user contacted StreamIndia's customer support. This is the frustration table. Every ticket represents a moment where something went wrong for the user — buffering, payment issues, login problems, content not available.In the real world, support tickets are a goldmine for churn prediction. A user who raises a cancellation request ticket is obviously about to leave. But even subtler signals matter — a user who raises 3 buffering complaints in one month is having a bad experience and is at risk.

1. This is the most thoughtful piece of logic in this table. Churned and active users don't complain about the same things.
Churned users complain more about Payment Failed and Cancellation Requests — because they're on their way out. Notice churned users have a 15% chance of raising a Cancellation Request ticket. Active users have 0% chance of raising a cancellation ticket — because they're not cancelling.
Active users complain more about Buffering and Content Not Available — these are technical frustrations from people who actually want to use the platform but are having issues.
This difference in complaint patterns is a real signal that data teams at OTT companies look for.

2. Not every user raises a support ticket. We make 60% of churned users raise tickets — because unhappy users complain before leaving. Only 20% of active users raise tickets — because satisfied users rarely contact support.
The continue keyword means — skip this user entirely, don't generate any tickets for them. This is why our table has only 9,529 rows instead of hundreds of thousands. Most users never contacted support.
num_tickets controls how many complaints each user raised. Churned users raised up to 4 tickets. Active users raised only 1 or 2. Multiple tickets from the same user is a strong churn signal.

3. This captures another real pattern — churned users had their issues resolved less often. Only 40% of churned users got their issue resolved, while 70% of active users did. This reflects two realities — either support failed to help them which caused them to churn, or they were already disengaged so they didn't follow up to get resolution. Either way, low resolution rate correlates with churn.

4. We record how long support took to respond — anywhere from 1 hour to 72 hours (3 days). In our EDA later we will check whether slow response times correlate with churn. In real OTT companies, slow support response is a known driver of cancellations.

5. Every table is linked by user_id. This is called a star schema — one central table with supporting tables around it. This is exactly how real OTT company databases are structured.

The One Line Summary of Table 4:
We created a support history where churned users complained more, got resolved less, and raised cancellation requests — because in real OTT platforms, support ticket patterns are one of the earliest and clearest warning signs that a user is about to leave.

The One Line Summary of the Entire Phase 1:
We didn't download a dataset. We architected one — with real Indian market patterns, real business logic, and real churn signals baked into every single table. That is what makes this project different from everyone else's.

# THE END OF PHASE 1 : DATABASE CREATION ! 